# Detecting objects with YOLOX in OpenCV

In this notebook, we use the popular YOLOX model in OpenCV to detect up to 80 objects via the webcam.

## Downloading the YOLOX model from Hugging Face

First, we need to download the YOLOX model from Hugging Face. The following cell downloads a file called `yolox_nano.onnx`. This file is a neural network model in ONNX format. ONNX stands for Open Neural Network eXchange and is one of the most popular formats for small models running in the browser or on hardware devices.

In [ ]:
import httpx
import os
from huggingface_hub import hf_hub_url

YOLOX_ONNX = "/tmp/yolox_nano.onnx"

response = httpx.get(hf_hub_url(repo_id="hr16/yolox-onnx", filename="yolox_nano.onnx"), follow_redirects=True)
response.raise_for_status()
with open(YOLOX_ONNX, 'wb') as f:
  f.write(response.content)
  print("Model downloaded")

## Loading the model

With the ONNX file downloaded, we now load the model into OpenCV using the `readNet` method in OpenCV's `dnn` module.

In [ ]:
import cv2

yolox_net = cv2.dnn.readNet(YOLOX_ONNX)
print("Neural network loaded")

## Defining the classes from the COCO dataset

The model doesn’t return the name of what it sees, only a number. To fix this, we define an array of strings called `COCO_CLASSES` which map these to their types. For example, if the model returns a 2, we can use this array to know that it’s detected a car.

In [ ]:
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush',
]

## The detect function

The code in the next cell defines a new function called `detect`. After we connect to the webcam, we'll run this function continuously, passing in frames from the webcam for the YOLOX model to detect.

There is a lot of detail in this detect function. If you are interested in seeing how it works, expand the cell to view the code!

In [ ]:
import numpy as np

INPUT_SIZE = 416

# YOLOX is "anchor-free": it makes one prediction per cell of three grids
# (52x52, 26x26 and 13x13 for a 416x416 input), each cell covering a square
# of 8, 16 or 32 pixels. These arrays hold each prediction's grid cell and
# cell size, in the same order as the network's 3549 output rows.
grid_cells = []
cell_sizes = []
for stride in (8, 16, 32):
    n = INPUT_SIZE // stride
    xs, ys = np.meshgrid(np.arange(n), np.arange(n))
    grid_cells.append(np.stack((xs, ys), axis=2).reshape(-1, 2))
    cell_sizes.append(np.full((n * n, 1), stride))
grid_cells = np.concatenate(grid_cells)
cell_sizes = np.concatenate(cell_sizes)

def detect(frame, min_confidence=0.4, overlap_threshold=0.45):
  """Run YOLOX-nano on a BGR frame. Returns a list of detections, each
  {"type", "x", "y", "w", "h", "confidence"} in frame pixels."""
  # 1. Letterbox: shrink to fit 416x416 keeping the aspect ratio, and pad
  #    the rest with grey. YOLOX takes raw BGR 0-255 pixels, no scaling.
  scale = min(INPUT_SIZE / frame.shape[0], INPUT_SIZE / frame.shape[1])
  resized = cv2.resize(frame, (int(frame.shape[1] * scale), int(frame.shape[0] * scale)))
  padded = np.full((INPUT_SIZE, INPUT_SIZE, 3), 114, dtype=np.uint8)
  padded[:resized.shape[0], :resized.shape[1]] = resized
  blob = cv2.dnn.blobFromImage(padded, 1.0, (INPUT_SIZE, INPUT_SIZE), swapRB=False)

  # 2. Run the network: 3549 rows of [cx, cy, w, h, objectness, 80 class scores].
  yolox_net.setInput(blob)
  output = yolox_net.forward()[0]

  # 3. Decode: box centres are offsets within a grid cell, sizes are
  #    log-scaled — both measured in cells. Convert to pixels.
  centres = (output[:, :2] + grid_cells) * cell_sizes
  sizes = np.exp(output[:, 2:4]) * cell_sizes
  boxes = np.concatenate([centres - sizes / 2, sizes], axis=1) / scale

  # 4. Score = "is there an object here?" x "is it this class?"
  class_scores = output[:, 4:5] * output[:, 5:]
  class_ids = class_scores.argmax(axis=1)
  confidences = class_scores.max(axis=1)

  # 5. Non-maximum suppression: drop boxes that overlap a more confident one.
  keep = cv2.dnn.NMSBoxes(boxes.tolist(), confidences.tolist(), min_confidence, overlap_threshold)

  height, width = frame.shape[:2]
  detections = []
  for i in np.array(keep).flatten():
    x, y, w, h = boxes[i]
    x0, y0 = max(0, int(x)), max(0, int(y))
    x1, y1 = min(width, int(x + w)), min(height, int(y + h))
    detections.append({
      'type': COCO_CLASSES[class_ids[i]],
      'x': x0, 'y': y0, 'w': x1 - x0, 'h': y1 - y0,
      'confidence': float(confidences[i]),
    })
  return detections

## Start the webcam!

Now that everything is setup, we'll start the webcam, capture frames in a loop, and pass them to the detect function. If the detect function sees anything in the frame, it will return an array of detections. We'll then pass these detections to the canvas to plot them on the screen.

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    detections = detect(frame)
    canvas.draw_bounding_boxes(detections)
finally:
  camera.stop()